<a href="https://colab.research.google.com/github/TatanPerez/4101134-Procesamiento-Digital-de-Imagenes/blob/main/Proyecto%20Final/License_plate_detection_HF_Space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Universidad Nacional de Colombia sede Manizales**

**Proyecto Final** - Procesamiento digital de imagenes **PDI**

**Estudiantes:**
- Wilmer Sebastian Perez Cuastumal - wiperezc@unal.edu.co
- Daniel Mauricio Mejia Hoyos - dmejiaho@unal.edu.co

**Profesor:** Lucas Iturriago - liturriago@unal.edu.co

**Monitora:** Isabella Valero Mora - lvalerom@unal.edu.co


Facultad de ingeniería y arquitectura

Departamento de ingeniería eléctrica, electrónica y computación

**Notas:** a continuacion estan link con informaion de donde se realizo el entrenamineto y exportacion a ExrcuTorch

- [Entrenamiento y exportacion a .pte en kaggle](https://www.kaggle.com/code/pereztatan/license-plate-detection)
- [Base de datos que se utilizo](https://universe.roboflow.com/augmented-startups/vehicle-registration-plates-trudk/dataset/1?ref=roboflow2huggingface)
- [Link para descargar base de datos](https://app.roboflow.com/ds/eIO478JpJz?key=ZtJRnSLMYl)

## 1. Servir Modelos de ExecuTorch en Hugging Face Spaces (Gradio)

Una vez que los modelos han sido exportados al formato optimizado de ExecuTorch (`.pte`), el siguiente paso es desplegarlos en producción para que puedan ser consumidos por aplicaciones externas. Hugging Face Spaces proporciona un entorno ideal para hospedar aplicaciones web interactivas de aprendizaje automático utilizando la librería **Gradio**.

### 1.1. Arquitectura de Despliegue en Hugging Face

En este laboratorio, construiremos un servidor Gradio que puede procesar tareas de:
1. **Detección de objetos**

Debido a que ExecuTorch requiere librerías nativas compiladas de C++ que pueden variar entre sistemas operativos, implementaremos un cargador robusto con un **mecanismo de fallback (contingencia)**. Si el entorno no dispone del runtime nativo de ExecuTorch (`executorch.runtime`), la aplicación cargará automáticamente los modelos base de PyTorch en FP32/FP16 para garantizar que la interfaz de usuario siga funcionando correctamente.


In [ ]:
import os
import sys
import shutil
import base64
import requests
from PIL import Image
from huggingface_hub import notebook_login
from google.colab import userdata

## 2. Creación y Configuración del Repositorio de Hugging Face

Para automatizar la subida y configuración del Space, primero debemos autenticarnos con Hugging Face. Esto abrirá una interfaz interactiva donde ingresarás tu **User Access Token** (con permisos de *write*).

In [ ]:
try:
    notebook_login()
except Exception as e:
    print("[INFO] notebook_login no está disponible o no se ejecuta en un entorno interactivo. Continúa con la generación de archivos locales.")

### 2.1. Configuración de Variables del Servidor

Define tu nombre de usuario de Hugging Face y el nombre que deseas darle a tu nuevo Space.

In [ ]:
HF_USER = "TatanPerez"
SPACE_NAME = "Car_and_motorcycle_license_plate_detection"
REPO_URL = f"https://huggingface.co/spaces/{HF_USER}/{SPACE_NAME}"

print(f"URL de clonación del Space: {REPO_URL}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2.2. Clonación y Configuración de Git LFS

Clonamos el repositorio del Space. Dado que subiremos archivos de modelos pesados (`.pte`), debemos configurar Git LFS (Large File Storage) para evitar rechazos en el comando `git push`.

In [ ]:
# 1. Clonar el repositorio del space creado
!git clone {REPO_URL}

In [ ]:
# 2. Configurar Git LFS para que gestione los modelos binarios .pte
%cd {SPACE_NAME}
!git lfs track "*.pte"
!git add .gitattributes
%cd ..

## 3. Generación de los Archivos de Despliegue (Gradio & Docker)

Escribiremos dinámicamente los tres archivos esenciales dentro del directorio de nuestro Space:
1.  `app.py`: El código de la aplicación de inferencia Gradio.
2.  `Dockerfile`: La definición de dependencias del contenedor de Docker.
3.  `README.md`: Los metadatos de configuración que Hugging Face requiere para entender el SDK y compilar el contenedor.

In [ ]:
# @title 1. Generar app.py con la aplicación Gradio en Float32
app_content = '''
import os
import sys
import numpy as np
import cv2
import torch
import gradio as gr
from PIL import Image
import pandas as pd
from datetime import datetime

# ── OCR ──────────────────────────────────────────────────────────────────────
try:
    import easyocr
    OCR_READER = easyocr.Reader(['en'], gpu=False, verbose=False)
    OCR_AVAILABLE = True
    print("[INFO] EasyOCR cargado correctamente.")
except ImportError:
    OCR_AVAILABLE = False
    print("[WARNING] EasyOCR no disponible. Instala con: pip install easyocr")

# ── ExecuTorch ────────────────────────────────────────────────────────────────
try:
    from executorch.runtime import Runtime, Program
    EXECUTORCH_AVAILABLE = True
    print("[INFO] Runtime de ExecuTorch cargado correctamente.")
except ImportError:
    EXECUTORCH_AVAILABLE = False
    print("[WARNING] ExecuTorch no está disponible. Usando fallback de PyTorch.")

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False
    print("[WARNING] Ultralytics no está disponible.")

PATH_MODEL_DET = "yolo8n_best.pte"
COCO_CLASSES = ['License_Plate']

np.random.seed(42)
COLOR_PALETTE = np.random.randint(0, 255, size=(100, 3), dtype=np.uint8)


# ── Preprocesamiento ──────────────────────────────────────────────────────────
def preprocesar_imagen(img_pil: Image.Image, size: tuple, normalizar: bool = True) -> torch.Tensor:
    if img_pil.mode != "RGB":
        img_pil = img_pil.convert("RGB")
    img_resized = img_pil.resize(size)
    img_np = np.array(img_resized, dtype=np.float32) / 255.0
    img_transposed = np.transpose(img_np, (2, 0, 1))
    tensor = torch.from_numpy(img_transposed).unsqueeze(0).contiguous()
    return tensor


# ── OCR sobre recortes de placa ───────────────────────────────────────────────
def extraer_texto_placas(recortes: list) -> list:
    textos = []
    if not OCR_AVAILABLE or not recortes:
        return textos
    for recorte in recortes:
        try:
            recorte_np = np.array(recorte)
            resultados = OCR_READER.readtext(recorte_np, detail=0, paragraph=True)
            texto = " ".join(resultados).strip().upper()
            texto_limpio = "".join(c for c in texto if c.isalnum() or c in ("-", " "))
            textos.append(texto_limpio if texto_limpio else "???")
        except Exception as e:
            print(f"[WARNING] Error OCR en recorte: {e}")
            textos.append("ERROR")
    return textos


# ── Postprocesamiento + dibujo ────────────────────────────────────────────────
def postprocesar_deteccion(boxes, scores, labels, original_img: Image.Image, threshold: float = 0.25):
    img_np = np.array(original_img)
    h, w, _ = img_np.shape

    boxes_np  = boxes.detach().numpy() if isinstance(boxes, torch.Tensor) else np.array(boxes)
    scores_np = scores.detach().numpy() if isinstance(scores, torch.Tensor) else np.array(scores)
    labels_np = labels.detach().numpy() if isinstance(labels, torch.Tensor) else np.array(labels)

    # Primera pasada: recortar placas para OCR
    recortes = []
    for box, score in zip(boxes_np, scores_np):
        if score >= threshold:
            xmin, ymin, xmax, ymax = int(box[0]), int(box[1]), int(box[2]), int(box[3])
            xmin = max(0, min(xmin, w - 1))
            ymin = max(0, min(ymin, h - 1))
            xmax = max(0, min(xmax, w - 1))
            ymax = max(0, min(ymax, h - 1))
            recortes.append(original_img.crop((xmin, ymin, xmax, ymax)))

    textos_ocr = extraer_texto_placas(recortes)

    # Segunda pasada: dibujar cajas y textos
    ocr_idx = 0
    for box, score, label_idx in zip(boxes_np, scores_np, labels_np):
        if score >= threshold:
            xmin, ymin, xmax, ymax = int(box[0]), int(box[1]), int(box[2]), int(box[3])
            xmin = max(0, min(xmin, w - 1))
            ymin = max(0, min(ymin, h - 1))
            xmax = max(0, min(xmax, w - 1))
            ymax = max(0, min(ymax, h - 1))

            color = [int(c) for c in COLOR_PALETTE[int(label_idx) % len(COLOR_PALETTE)]]
            cv2.rectangle(img_np, (xmin, ymin), (xmax, ymax), color, 3)

            clase_texto = f"{COCO_CLASSES[int(label_idx) % len(COCO_CLASSES)]}: {score:.2f}"
            cv2.putText(img_np, clase_texto, (xmin, max(ymin - 25, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if ocr_idx < len(textos_ocr):
                cv2.putText(img_np, textos_ocr[ocr_idx], (xmin, max(ymin - 8, 30)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
            ocr_idx += 1

    return Image.fromarray(img_np), recortes, textos_ocr


# ── Model Runner ──────────────────────────────────────────────────────────────
class ModelRunner:
    def __init__(self, pte_path: str, fallback_model_fn):
        self.use_executorch = EXECUTORCH_AVAILABLE and os.path.exists(pte_path)
        if self.use_executorch:
            print(f"[INFO] Cargando modelo ExecuTorch: {pte_path}")
            try:
                self.runtime = Runtime.get()
                self.program = self.runtime.load_program(pte_path)
                self.method  = self.program.load_method("forward")
            except Exception as e:
                print(f"[ERROR] Error al cargar ExecuTorch: {e}. Usando fallback.")
                self.use_executorch = False

        if not self.use_executorch:
            print(f"[INFO] Cargando fallback PyTorch para: {pte_path}")
            try:
                self.model = fallback_model_fn()
                if hasattr(self.model, "eval"):
                    try:    self.model = self.model.eval()
                    except: pass
                if hasattr(self.model, "model") and hasattr(self.model.model, "eval"):
                    try:    self.model.model.eval()
                    except: pass
            except Exception as e:
                print(f"[ERROR] Error al cargar fallback: {e}")
                self.model = None

    def run(self, input_tensor: torch.Tensor):
        if self.use_executorch:
            outputs = self.method.execute((input_tensor,))
            return outputs[0] if isinstance(outputs, list) and len(outputs) == 1 else outputs
        else:
            with torch.no_grad():
                return self.model(input_tensor)


runner_det = ModelRunner(PATH_MODEL_DET, lambda: YOLO("yolo26x.pte") if YOLO_AVAILABLE else None)

# ── Historial ─────────────────────────────────────────────────────────────────
history_columns = ["Fecha", "Imagen", "Confidence", "IoU", "Detecciones", "Placas"]


# ── Función principal ─────────────────────────────────────────────────────────
def predict_detection(image: Image.Image, confidence_threshold: float, iou_threshold: float, history_data: list):

    empty_df = pd.DataFrame(history_data, columns=history_columns)

    if image is None:
        return None, empty_df, history_data

    # ── Normalizar imagen (cámara o archivo) ──────────────────────────────────
    # Con type="pil" Gradio siempre entrega PIL. mirror_webcam=False evita el
    # espejo, así que NO hace falta flip adicional. Solo garantizamos RGB.
    if image.mode != "RGB":
        image = image.convert("RGB")

    try:
        if not (runner_det.use_executorch or
                (hasattr(runner_det, "model") and runner_det.model is not None)):
            img_np = np.array(image)
            cv2.putText(img_np, "Modelo de deteccion no cargado",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            return Image.fromarray(img_np), empty_df, history_data

        # ── EXECUTORCH ────────────────────────────────────────────────────────
        if runner_det.use_executorch:
            tensor = preprocesar_imagen(image, (640, 640), normalizar=False)
            output = runner_det.run(tensor)

            pred        = output[0].numpy() if isinstance(output, torch.Tensor) else output[0]
            predictions = pred[0].T if len(pred.shape) == 3 else pred.T
            boxes       = predictions[:, :4]
            scores      = predictions[:, 4:]
            max_scores  = np.max(scores, axis=1)
            class_ids   = np.argmax(scores, axis=1)

            keep               = max_scores >= confidence_threshold
            filtered_boxes     = boxes[keep]
            filtered_scores    = max_scores[keep]
            filtered_class_ids = class_ids[keep]

            if len(filtered_boxes) > 0:
                cx, cy, w_box, h_box = (filtered_boxes[:, 0], filtered_boxes[:, 1],
                                        filtered_boxes[:, 2], filtered_boxes[:, 3])
                orig_w, orig_h = image.size  # PIL: (width, height)
                scale_x = orig_w / 640.0
                scale_y = orig_h / 640.0

                x1 = (cx - w_box / 2) * scale_x
                y1 = (cy - h_box / 2) * scale_y
                x2 = (cx + w_box / 2) * scale_x
                y2 = (cy + h_box / 2) * scale_y

                boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1)
                boxes_xywh = np.stack([x1, y1, w_box * scale_x, h_box * scale_y], axis=1)

                indices = cv2.dnn.NMSBoxes(
                    boxes_xywh.tolist(), filtered_scores.tolist(),
                    confidence_threshold, iou_threshold
                )

                if len(indices) > 0:
                    indices = np.array(indices).flatten()

                    result_image, recortes, textos_ocr = postprocesar_deteccion(
                        boxes_xyxy[indices],
                        filtered_scores[indices],
                        filtered_class_ids[indices],
                        image,
                        threshold=confidence_threshold
                    )

                    placas_str = " | ".join(textos_ocr) if textos_ocr else "—"
                    history_data = history_data + [[
                        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        f"Imagen_{len(history_data)+1}",
                        round(confidence_threshold, 2),
                        round(iou_threshold, 2),
                        int(len(indices)),
                        placas_str
                    ]]
                    return result_image, pd.DataFrame(history_data, columns=history_columns), history_data

            # Sin detecciones
            history_data = history_data + [[
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                f"Imagen_{len(history_data)+1}",
                round(confidence_threshold, 2),
                round(iou_threshold, 2),
                0,
                "Sin detecciones"
            ]]
            return image, pd.DataFrame(history_data, columns=history_columns), history_data

        # ── YOLO PYTORCH ──────────────────────────────────────────────────────
        results = runner_det.model(image, conf=confidence_threshold, iou=iou_threshold, verbose=False)
        r = results[0]

        boxes  = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()
        labels = r.boxes.cls.cpu().numpy()

        result_image, recortes, textos_ocr = postprocesar_deteccion(
            boxes, scores, labels, image, threshold=confidence_threshold
        )

        placas_str = " | ".join(textos_ocr) if textos_ocr else "—"
        history_data = history_data + [[
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            f"Imagen_{len(history_data)+1}",
            round(confidence_threshold, 2),
            round(iou_threshold, 2),
            int(len(boxes)),
            placas_str
        ]]
        return result_image, pd.DataFrame(history_data, columns=history_columns), history_data

    except Exception as e:
        img_np = np.array(image)
        cv2.putText(img_np, f"Error: {str(e)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        return Image.fromarray(img_np), pd.DataFrame(history_data, columns=history_columns), history_data


# ── UI Gradio ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="YOLO Object Detection System") as demo:

    gr.Markdown("""
    # 🚗 YOLO Object Detection System
    ### Detección de placas con ExecuTorch + OCR automático
    """)

    history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📥 Entrada")
            img_in_det = gr.Image(
                type="pil",
                label="Sube una imagen o usa la cámara",
                sources=["upload", "webcam"]   # ✅ lista correcta
            )
            gr.Markdown("### ⚙️ Configuración")
            confidence = gr.Slider(0.1, 1.0, value=0.25, label="Confidence Threshold")
            iou        = gr.Slider(0.1, 1.0, value=0.7,  label="IoU Threshold")

            if runner_det.use_executorch:
                status_text = "🟢 ExecuTorch cargado"
            elif runner_det.model is not None:
                status_text = "🟢 PyTorch cargado"
            else:
                status_text = "🔴 Error al cargar modelo"

            ocr_status = "🟢 OCR activo (EasyOCR)" if OCR_AVAILABLE else "🟡 OCR no disponible"
            gr.Markdown(status_text + "  \\n" + ocr_status)

            btn_run_det = gr.Button("🚀 Detectar y Leer Placa", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("## 🧾 Resultado")
            img_out_det = gr.Image(type="pil", label="Detecciones + Texto OCR")

    gr.Markdown("## 📊 Historial de Inferencias")
    history_table = gr.Dataframe(
        headers=history_columns,
        datatype=["str", "str", "number", "number", "number", "str"],
        value=[],
        interactive=False
    )

    btn_run_det.click(
        fn=predict_detection,
        inputs=[img_in_det, confidence, iou, history_state],
        outputs=[img_out_det, history_table, history_state]
    )

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, show_error=True, theme=gr.themes.Soft() )
'''

os.makedirs(SPACE_NAME, exist_ok=True)

with open(f"{SPACE_NAME}/app.py", "w", encoding="utf-8") as f:
    f.write(app_content)

print("app.py creado con éxito en el space.")

In [ ]:
# @title 2. Generar Dockerfile con el entorno de compilación de ExecuTorch
dockerfile_content = """
FROM python:3.10-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    libgl1 \
    libglib2.0-0 \
    git \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
    PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir --quiet \
    executorch \
    torch==2.11.0 \
    torchvision \
    numpy \
    opencv-python-headless \
    gradio \
    gradio_client \
    ultralytics \
    easyocr

COPY --chown=user app.py ./app.py
COPY --chown=user *.pte ./

EXPOSE 7860

CMD ["python", "app.py"]
"""

with open(f"{SPACE_NAME}/Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile_content)
print("Dockerfile creado con éxito en el space.")

In [ ]:
# @title 3. Generar README.md con metadatos YAML correctos para Hugging Face Spaces
readme_content = f"""---
title: {SPACE_NAME}
emoji: 🚀
colorFrom: blue
colorTo: green
sdk: docker
app_file: app.py
pinned: false
---

# Inferencia de Visión con ExecuTorch (Float32)
Despliegue interactivo de clasificación, segmentación y detección con Gradio y Docker.
"""

with open(f"{SPACE_NAME}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)
print("README.md creado con éxito en el space.")

### 3.1. Copiado de los Modelos `.pte`

Copiamos los archivos de los modelos exportados dentro de la carpeta del repositorio local del Space.

In [ ]:
# Copiar modelos exportados al directorio del space
for model_file in ["yolo8n_best.pte"]:
    if os.path.exists(model_file):
        shutil.copy(model_file, f"{SPACE_NAME}/")
        print(f"Modelo copiado: {model_file}")
    else:
        print(f"[WARNING] No se encontró el modelo local '{model_file}'. Recuerda exportarlo primero.")

## 4. Publicación y Despliegue en el Space

Configuramos el nombre de usuario de Git, realizamos el commit con todos los archivos agregados y enviamos los cambios para activar la compilación del contenedor Docker en Hugging Face.

In [ ]:
%cd {SPACE_NAME}
!git config --global user.email "tu_correo@unal.edu.co"
!git config --global user.name "tu_usuario_hf"
!git add .
!git commit -m "Despliegue inicial de modelos ExecuTorch FP32 en Hugging Face Spaces"
!git push
%cd ..

## 5. Pruebas de Cliente (Consumo del API del Space)

Cuando el contenedor finalice de compilarse en Hugging Face, la API web estará expuesta. Podemos enviar solicitudes REST desde Python.

### 5.1. `gradio_client`

Método recomendado para interactuar directamente con la API desde Python.

In [ ]:
from gradio_client import Client, handle_file

hf_token = userdata.get("HuggingFace")
client = Client(f"{HF_USER}/{SPACE_NAME}", hf_token=hf_token)
client.view_api()
# Envolver la ruta local con handle_file()
result = client.predict(
    image=handle_file("/content/placa 3.jpg"),  # Subir una imagen y reemplazar ruta de acceso
    api_name="/predict_detection"   # o el que te muestre view_api()
)

print(result)